# Part 2: target vs distractor connectivity matrices (ds002680 go/no-go)

| **target (go) / distractor (no-go)** |
 **pre (−0.5 to −0.01 s) / post (0.01 to 0.5 s)** |

All channel pairs among `chans`, alpha and beta, four methods. Effect = post − pre per condition; the comparison of interest is target effect − distractor effect, so differences that already exist before the stimulus cancel.

**Note:** targets include a button release (~330 ms) inside the post window; distractors do not. A target > distractor difference can be motor, not perceptual.

In [ ]:
# re-read edited .py files in src/bids_conn on every cell run
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns

from bids_conn import find_recordings, get_paths, load_raw, make_epochs, stimulus_metadata
from bids_conn.connectivity import subsample, window_connectivity

mne.set_log_level("WARNING")

paths = get_paths()
out_dir = paths["results"] / "part2_target_distractor"      # one CSV per recording goes here
out_dir.mkdir(parents=True, exist_ok=True)
print(paths["bids_root"], paths["bids_root"].exists())

## Config

In [ ]:
recs = find_recordings(paths["bids_root"], task="gonogo")
display(recs[["subject", "session", "run", "rec_id"]])
rec_ids = list(recs["rec_id"])        # all recordings; or e.g. rec_ids = rec_ids[:1] to test

# preprocessing (continuous data, before epoching)
l_freq, h_freq = 1.0, 40.0
reference      = 'average'            # original 'common' reference sits near C3/C4 and inflates their coupling
resample_to    = 500

# epochs
epoch_tmin, epoch_tmax = -0.6, 1.0    # stimuli are >= 1.8 s apart, so epochs never overlap
chans      = ['F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2']
reject_uv  = 100                      # peak-to-peak, checked on `chans` only (FP1/FP2 carry the blinks)
correct_only = True                   # targets with a response, distractors without
conditions = ['target', 'distractor']
equalize_trials = True                # same n per condition (PLV/PLI/wPLI depend on trial count)
seed = 0

# part 2 analysis: every pair among `chans`
windows = {'pre': (-0.5, -0.01), 'post': (0.01, 0.5)}      # 246 samples each at 500 Hz
bands   = {'alpha': (8, 13), 'beta': (13, 30)}

con_kwargs = dict(
    method=['plv', 'pli', 'wpli', 'wpli2_debiased'],
    mode='multitaper',
    mt_bandwidth=4,                   # +/-2 Hz, 1 taper
    mt_adaptive=False,
    n_jobs=1,
    verbose=False,
)
overwrite = False

## Pipeline for one recording

In [ ]:
def run_recording(rec):
    """Load one recording, compute all-pairs connectivity per condition and window, return result rows."""
    raw = load_raw(rec.path, l_freq=l_freq, h_freq=h_freq, sfreq=resample_to, reference=reference)
    meta = stimulus_metadata(rec.path)
    epochs, n_rejected = make_epochs(raw, meta, tmin=epoch_tmin, tmax=epoch_tmax,
                                     reject_uv=reject_uv, reject_picks=chans)
    if correct_only:
        epochs = epochs[epochs.metadata["correct"].to_numpy()]
    epochs = epochs.pick(chans)

    by_cond = {cond: epochs[cond] for cond in conditions}
    if equalize_trials:
        rng = np.random.default_rng(seed)
        n_min = min(len(ep) for ep in by_cond.values())
        by_cond = {cond: subsample(ep, n_min, rng) for cond, ep in by_cond.items()}

    rows = []
    for cond, ep in by_cond.items():
        for row in window_connectivity(ep, windows, None, bands, con_kwargs):   # None = all pairs
            row.update(condition=cond, subject=rec.subject, session=rec.session, run=rec.run,
                       rec_id=rec.rec_id, n_trials=len(ep), n_rejected=n_rejected)
            rows.append(row)
    return rows

## Run all recordings

In [ ]:
failed = []
todo = recs[recs["rec_id"].isin(rec_ids)]
for i, rec in enumerate(todo.itertuples(), start=1):
    out_file = out_dir / f"{rec.rec_id}.csv"
    if out_file.exists() and not overwrite:
        print(f"[{i}/{len(todo)}] {rec.rec_id}: already done, skipping")
        continue

    try:
        rows = run_recording(rec)
    except Exception as e:
        print(f"[{i}/{len(todo)}] {rec.rec_id}: FAILED - {type(e).__name__}: {e}")
        failed.append(rec.rec_id)
        continue

    pd.DataFrame(rows).to_csv(out_file, index=False)
    print(f"[{i}/{len(todo)}] {rec.rec_id}: done, {len(rows)} rows")

print("failed:", failed)

## Combine
Runs are averaged within subject, then every comparison is a column: `<condition>_<window>`, the two effects (post − pre) and their difference.

In [ ]:
df = pd.concat([pd.read_csv(out_dir / f"{r}.csv", dtype={"subject": str, "session": str, "run": str})
                for r in rec_ids if (out_dir / f"{r}.csv").exists()], ignore_index=True)
df.to_csv(paths["results"] / "part2_target_distractor_all_recordings.csv", index=False)

subj = df.groupby(["subject", "condition", "window", "method", "band", "ch1", "ch2"], as_index=False)["value"].mean()
n_subjects = subj["subject"].nunique()
print(df.shape, "| recordings:", df["rec_id"].nunique(), "| subjects:", n_subjects)

wide = subj.pivot_table(index=["subject", "method", "band", "ch1", "ch2"], columns=["condition", "window"], values="value")
wide.columns = [f"{c}_{w}" for c, w in wide.columns]                    # e.g. 'target_post'
wide["target_effect"]        = wide["target_post"] - wide["target_pre"]
wide["distractor_effect"]    = wide["distractor_post"] - wide["distractor_pre"]
wide["target_vs_distractor"] = wide["target_effect"] - wide["distractor_effect"]
wide = wide.reset_index()
wide.head()

## Heatmaps (mean over subjects)
One figure per band. Rows: target | distractor | target − distractor. Columns: pre | post | post − pre. The bottom-right panel is the comparison of interest.

In [ ]:
def to_matrix(table, column):
    """Mean over subjects of one `wide` column as a symmetric chans x chans DataFrame (diagonal NaN)."""
    m = table.pivot_table(index="ch1", columns="ch2", values=column).reindex(index=chans, columns=chans)
    return m.combine_first(m.T)          # copy each pair into the other triangle


method = 'wpli2_debiased'                # or 'wpli', 'pli', 'plv'

for band in bands:
    sel = wide[(wide["method"] == method) & (wide["band"] == band)]
    grid = [
        [to_matrix(sel, "target_pre"),     to_matrix(sel, "target_post"),     to_matrix(sel, "target_effect")],
        [to_matrix(sel, "distractor_pre"), to_matrix(sel, "distractor_post"), to_matrix(sel, "distractor_effect")],
        [to_matrix(sel, "target_pre") - to_matrix(sel, "distractor_pre"),
         to_matrix(sel, "target_post") - to_matrix(sel, "distractor_post"),
         to_matrix(sel, "target_vs_distractor")],
    ]
    row_names = ['target', 'distractor', 'target − distractor']
    col_names = ['pre', 'post', 'post − pre']

    # raw values share one scale; every difference panel shares one symmetric scale
    raw_panels  = [grid[r][c] for r in range(2) for c in range(2)]
    diff_panels = [grid[r][2] for r in range(3)] + [grid[2][c] for c in range(2)]
    rmax = max(abs(x).max().max() for x in raw_panels)
    dmax = max(abs(x).max().max() for x in diff_panels)
    raw_vmin, raw_cmap = (-rmax, 'RdBu_r') if method == 'wpli2_debiased' else (0, 'Blues')   # debiased can be negative

    fig, axes = plt.subplots(3, 3, figsize=(16, 14))
    for r in range(3):
        for c in range(3):
            if r == 2 or c == 2:
                sns.heatmap(grid[r][c], ax=axes[r, c], cmap='RdBu_r', vmin=-dmax, vmax=dmax, center=0,
                            annot=True, fmt='.2f', square=True)
            else:
                sns.heatmap(grid[r][c], ax=axes[r, c], cmap=raw_cmap, vmin=raw_vmin, vmax=rmax,
                            annot=True, fmt='.2f', square=True)
            axes[r, c].set_title(f'{row_names[r]}: {col_names[c]}')

    fig.suptitle(f'ds002680, {method}, {band}, mean of {n_subjects} subject(s)')
    plt.tight_layout()
    plt.show()

## Group statistics (one test per edge)
Wilcoxon signed-rank across subjects on each comparison column, FDR-corrected within each method × band × comparison. Needs enough subjects to mean anything; with fewer than 6 it only reports that.

In [ ]:
import pingouin as pg
from statsmodels.stats.multitest import multipletests

comparisons = ["target_effect", "distractor_effect", "target_vs_distractor"]

if n_subjects < 6:
    print(f"Only {n_subjects} subject(s): download more of ds002680 before testing.")
else:
    results = []
    for (method, band, ch1, ch2), g in wide.groupby(["method", "band", "ch1", "ch2"]):
        for comparison in comparisons:
            x = g[comparison].dropna()
            res = pg.wilcoxon(x)                       # H0: median difference = 0
            results.append({"method": method, "band": band, "ch1": ch1, "ch2": ch2, "comparison": comparison,
                            "median_diff": x.median(), "p": res["p-val"].iloc[0], "n": len(x)})
    stats = pd.DataFrame(results)
    stats["p_fdr"] = stats.groupby(["method", "band", "comparison"])["p"].transform(
        lambda p: multipletests(p, method="fdr_bh")[1])
    stats.to_csv(paths["results"] / "part2_edge_stats.csv", index=False)
    display(stats.query("method == 'wpli2_debiased'").sort_values("p_fdr").head(20))